In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import os
os.chdir("/content")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
# !rm -rf "/content/vlm-safety-reasoning"

In [5]:
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"

def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...
>>> Secrets loaded successfully.
>>> Configuring Git identity...
>>> Cloning repo...
>>> Working directory: /content/vlm-safety-reasoning
>>> Copying .env into local workspace...
>>> Installing requirements...
>>> Setup complete.


In [6]:
import json
from pathlib import Path
from data.loader import load_processed_dataset
from models.model_loader import load_model_for_inference
from models.inference import run_inference_batched
from core.run_manifest import save_run_manifest
from core.config import load_config
from core.io import get_drive_path, ensure_dir

In [14]:
MODEL_TIER = "2b"
N_SAMPLES = None
BATCH_SIZE = 32

DRIVE_RESULTS_DIR = get_drive_path("results", f"baseline_test_full_{MODEL_TIER}")
ensure_dir(DRIVE_RESULTS_DIR)
JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

In [15]:
base_config = load_config(task="unified")

In [16]:
run_config = {
    "experiment": "baseline_inference_full_testset",
    "model_tier": MODEL_TIER,
    "n_samples": N_SAMPLES,
    "batch_size": BATCH_SIZE,
    "max_new_tokens": base_config.get("max_new_tokens", 1000),
    "notes": "Pass 3: Final cleanup of the massive 4K outliers.",
    "full_yaml_state": base_config
}
save_run_manifest(str(DRIVE_RESULTS_DIR), run_config)

2026-07-17 14:49:45 | INFO     | core.run_manifest:save_run_manifest:38 - Run manifest saved to /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/run_manifest.json


'/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/run_manifest.json'

In [10]:
print("Loading dataset...")
splits = load_processed_dataset()
test_data = splits["test"]

Loading dataset...
2026-07-17 14:07:22 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-17 14:07:56 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-17 14:07:56 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-17 14:07:56 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples


In [11]:
img_first = test_data[0]["image"]
img_last = test_data[-1]["image"]

print(f"Smallest Image (Batch 1): {img_first.width}x{img_first.height} pixels")
print(f"Largest Image (Final Batch): {img_last.width}x{img_last.height} pixels")

Smallest Image (Batch 1): 320x240 pixels
Largest Image (Final Batch): 4416x3312 pixels


In [12]:
print("Loading model into VRAM...")
model, tokenizer, info = load_model_for_inference(tier=MODEL_TIER)

Loading model into VRAM...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-07-17 14:08:31 | INFO     | models.model_loader:load_model_for_inference:164 - Loading model for inference: unsloth/Qwen3-VL-2B-Instruct with max_seq_length=8192
==((====))==  Unsloth 2026.7.3: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [17]:
# Inference
print(f"Starting batched inference. Results will stream to: {JSONL_OUTPUT_PATH}")
results = run_inference_batched(
    model=model,
    tokenizer=tokenizer,
    dataset=test_data,
    batch_size=run_config["batch_size"],
    max_new_tokens=run_config["max_new_tokens"],
    max_samples=run_config["n_samples"],
    output_path=JSONL_OUTPUT_PATH
)
print("Inference Complete!")

Starting batched inference. Results will stream to: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/predictions.jsonl
2026-07-17 14:50:12 | INFO     | models.inference:run_inference_batched:263 - Auto-Resume: Found 2728 completed images in /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/predictions.jsonl


Filter:   0%|          | 0/3004 [00:00<?, ? examples/s]

2026-07-17 14:50:23 | INFO     | models.inference:run_inference_batched:270 - Remaining images to process: 276


Batched Inference: 100%|██████████| 9/9 [11:16<00:00, 75.18s/it] 

2026-07-17 15:01:40 | INFO     | models.inference:run_inference_batched:326 - Batched inference complete: 276 new samples processed.
Inference Complete!


## Evaluation

In [18]:
import os
import json
from evaluation.evaluator import run_full_evaluation

METRICS_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "metrics.json")
FAILURES_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "json_schema_failures.jsonl")
PARSED_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "parsed_predictions.jsonl")

In [19]:
print(f"Loading perfect predictions from {JSONL_OUTPUT_PATH}...")
raw_predictions = []
references = []

with open(JSONL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue

        try:
            record = json.loads(line)
            raw_predictions.append(record.get("raw_output", ""))
            references.append(record["sample"])
        except json.JSONDecodeError:
            pass
print(f"Loaded exactly {len(raw_predictions)} images.")

Loading perfect predictions from /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/predictions.jsonl...
Loaded exactly 3004 images.


In [20]:
final_eval_payload = run_full_evaluation(raw_predictions, references)
print(f"\nWrapper executed successfully! Total metrics calculated: {len(final_eval_payload['metrics'])}")

2026-07-17 15:14:26 | INFO     | evaluation.evaluator:run_full_evaluation:24 - Starting full evaluation pipeline...
2026-07-17 15:14:26 | INFO     | evaluation.evaluator:run_full_evaluation:71 - Computing captioning metrics...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

2026-07-17 15:14:37 | INFO     | evaluation.metrics_reasoning:batch_score_reasoning:59 - Computing global reasoning metrics over 170 valid reasons...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

2026-07-17 15:14:39 | INFO     | evaluation.metrics_reasoning:batch_score_reasoning:71 - Computing reasoning metrics for rule_1 over 170 valid reasons...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

2026-07-17 15:14:39 | INFO     | evaluation.evaluator:run_full_evaluation:94 - Evaluation complete. 1757 schema failures logged.

Wrapper executed successfully! Total metrics calculated: 80


In [21]:
with open(METRICS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_eval_payload["metrics"], f, indent=2, ensure_ascii=False)
with open(FAILURES_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for failure in final_eval_payload.get("failures", []):
        f.write(json.dumps(failure) + "\n")
with open(PARSED_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for pred in final_eval_payload.get("parsed_predictions", []):
        if pred is not None:
            f.write(json.dumps(pred) + "\n")
print(f"All artifacts perfectly saved to: {DRIVE_RESULTS_DIR}")

All artifacts perfectly saved to: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b
